In [65]:
import pandas as pd
import json
import sklearn
import numpy

In [9]:
#set project root
import os
os.chdir("/Users/Patron/Github/Spotify-Song-Recommendation/")

## Data Reader Logic

In [ ]:
#data reading logic
with open('dataset/data/mpd.slice.0-999.json', 'r') as f:
    data = json.load(f)

data    

In [18]:
#accessing keys
data.keys()

dict_keys(['info', 'playlists'])

In [17]:
#info stores the creation date, playlist slice and version
data['info']

{'generated_on': '2017-12-03 08:41:42.057563',
 'slice': '0-999',
 'version': 'v1'}

In [20]:
#holds all of the playlists
len(data['playlists'])

1000

In [ ]:
data['playlists'][2]

In [148]:
#first playlist
data['playlists'][0]

{'name': 'Throwbacks',
 'collaborative': 'false',
 'pid': 0,
 'modified_at': 1493424000,
 'num_tracks': 52,
 'num_albums': 47,
 'num_followers': 1,
 'tracks': [{'pos': 0,
   'artist_name': 'Missy Elliott',
   'track_uri': 'spotify:track:0UaMYEvWZi0ZqiDOoHU3YI',
   'artist_uri': 'spotify:artist:2wIVse2owClT7go1WT98tk',
   'track_name': 'Lose Control (feat. Ciara & Fat Man Scoop)',
   'album_uri': 'spotify:album:6vV5UrXcfyQD1wu4Qo2I9K',
   'duration_ms': 226863,
   'album_name': 'The Cookbook'},
  {'pos': 1,
   'artist_name': 'Britney Spears',
   'track_uri': 'spotify:track:6I9VzXrHxO9rA9A5euc8Ak',
   'artist_uri': 'spotify:artist:26dSoYclwsYLMAKD3tpOr4',
   'track_name': 'Toxic',
   'album_uri': 'spotify:album:0z7pVBGOD7HCIB7S8eLkLI',
   'duration_ms': 198800,
   'album_name': 'In The Zone'},
  {'pos': 2,
   'artist_name': 'Beyoncé',
   'track_uri': 'spotify:track:0WqIKmW4BTrj3eJFmnCKMv',
   'artist_uri': 'spotify:artist:6vWDO969PvNqNYHIOW5v0m',
   'track_name': 'Crazy In Love',
   'alb

In [ ]:
#each playlist has certain metadata, and a list of tracks
data['playlists'][0]['tracks']

In [ ]:
print(len(data['playlists'][0]['tracks']))
for track in data['playlists'][0]['tracks']:
    print(track['track_uri'])

In [ ]:
#can safely remove spotify:track: and be left with a unique track id

In [ ]:
for i in range(len(data["playlists"])):
    print(data["playlists"][i]["pid"])

## Barebones KNN

The problem statement is to create a model that can predict which tracks to put on a playlist, given the past history.
The plan is to store each playlist data (pid,track_uri).  We need to process the data from the complex json to a list of tuples saved to disk.

In [ ]:
def read_slices(file_path:string) -> Any:
    try:
        with open(file_path, 'r') as f:
           data = json.load(f)
           return data 
    except:
        raise FileError("Wrong File Path")

def extract_tracks(jsondata: Any) -> List:
    if "playlists" not in jsondata.keys():
        raise KeyError("Key not found")
    try:
        data = []
        #1000 playlists in one slice
        for i in range(1000):
            for track in jsondata['playlists'][i]['tracks']:
                # print(f'{i} - {track['track_uri'][14:]}')
                data.append([i,track['track_uri'][14:]])
        return data
    except:
        raise TypeError("Something bad happened :(")    


#map each track_uri to a integer, and persist them on disk
def encode_tracks(interactions):
    """
    interactions: list of [playlist_id, track_uri]

    Returns:
        encoded_interactions: list of (playlist_id, track_idx)
        track_to_idx: dict mapping track_uri -> track_idx
        idx_to_track: dict mapping track_idx -> track_uri
    """
    track_to_idx = {}
    idx_to_track = {}

    track_counter = 0
    encoded_interactions = []

    #map each track to a integer, and return the new list and maps
    for playlist_id, track_uri in interactions:
        if track_uri not in track_to_idx:
            track_to_idx[track_uri] = track_counter
            idx_to_track[track_counter] = track_uri
            track_counter += 1

        encoded_interactions.append(
            (playlist_id, track_to_idx[track_uri])
        )

    return encoded_interactions, track_to_idx, idx_to_track

from scipy.sparse import csr_matrix
def build_matrix(encoded_interactions,track_to_idx):
    rows = []
    cols = []
    data = []

    for p_id, t_idx in encoded_interactions:
        rows.append(p_id)
        cols.append(t_idx)
        data.append(1)

    X = csr_matrix(
        (data, (rows, cols)),
        shape=(max(rows) + 1, len(track_to_idx))
    )

    return X
        

In [123]:
jsondata = read_slices("dataset/data/mpd.slice.0-999.json")
track_data = extract_tracks(jsondata)
encoded_track_data,track_to_id,id_to_track = encode_tracks(track_data)
X = build_matrix(encoded_track_data,track_to_id)

In [124]:
X[2]

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 64 stored elements and shape (1, 34443)>

In [125]:
X_items = X.T

In [126]:
from sklearn.neighbors import NearestNeighbors

knn_model = NearestNeighbors(
    n_neighbors=5, 
    metric='cosine',
    algorithm='brute'
)
knn_model.fit(X_items)


,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'brute'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None


In [134]:
def encode_track_details(jsondata, num_playlists=1000):
    """
    Extract track details from playlists.

    Args:
        jsondata: the JSON data containing playlists
        num_playlists: how many playlists to process

    Returns:
        track_to_details: dict mapping track_uri -> (track_name, track_artist)
        playlist_tracks: dict mapping playlist_idx -> list of track_uris
    """
    track_to_details = {}   # URI -> (name, artist)
    playlist_tracks = {}    # playlist index -> list of track URIs

    for i in range(num_playlists):
        playlist_tracks[i] = []
        for track in jsondata['playlists'][i]['tracks']:
            track_uri = track['track_uri'][14:]  # remove 'spotify:track:' prefix
            track_name = track['track_name']
            track_artist = track['artist_name']

            playlist_tracks[i].append(track_uri)

            # store track details only once
            if track_uri not in track_to_details:
                track_to_details[track_uri] = (track_name, track_artist)

    return track_to_details, playlist_tracks


In [135]:
def decode_track_details(track_uri, track_to_details):
    return track_to_details.get(track_uri, (None, None))


In [143]:
track_to_details, playlist_trcks = encode_track_details(jsondata, num_playlists=1000)

track_uri = playlist_tracks[0][0]  # first track in first playlist
name, artist = decode_track_details(track_uri, track_to_details)

print(track_uri, name, artist)

0UaMYEvWZi0ZqiDOoHU3YI Lose Control (feat. Ciara & Fat Man Scoop) Missy Elliott


In [144]:
seed_track_id = playlist_tracks[0][5] 
seed_idx = track_to_id[seed_track_id]  

#given a seed track, find its closest neighbours and return them
distances, neighbors = knn_model.kneighbors(X_items[seed_idx], n_neighbors=20)

# decode indices back to track URIs
recommended_tracks = [id_to_track[idx] for idx in neighbors[0]]
recommended_tracks

['0XUfyU2QviPAs6bxSpXYG4',
 '1V4jC0vJ5525lEF1bFgPX2',
 '7xYnUQigPoIDAMPVK79NEq',
 '7iL6o9tox1zgHpKUfh9vuC',
 '5KY7zgFeH2GWoL1zP9mME6',
 '7uKcScNXuO3MWw6LowBjW1',
 '70cTMpcgWMcR18t9MRJFjB',
 '2CEgGE6aESpnmtfiZwYlbV',
 '4E5P1XyAFtrjpiIxkydly4',
 '0WqIKmW4BTrj3eJFmnCKMv',
 '2IpGdrWvIZipmaxo1YRxw5',
 '0CAfXk7DXMnon4gLudAp7J',
 '3VzJE6yGuj8fDExUh6TLnc',
 '0k6HUzaRHpQ3eEWr1C7Esh',
 '557un1HgwYMuqfWGSTmnxw',
 '5Q0Nhxo0l2bP3pNjpGJwV1',
 '1ARJhjuI6TNYZCxYygFQ4F',
 '5i66xrvSh1MjjyDd6zcwgj',
 '1dzQoRqT5ucxXVaAhTcT0J',
 '6RcQOut9fWL6FSqeIr5M1r']

In [147]:
for track_uri in recommended_tracks:
    name,artist = decode_track_details(track_uri,track_to_details)
    print(track_uri, name, artist)

0XUfyU2QviPAs6bxSpXYG4 Yeah! Usher
1V4jC0vJ5525lEF1bFgPX2 Shots LMFAO
7xYnUQigPoIDAMPVK79NEq Run It! Chris Brown
7iL6o9tox1zgHpKUfh9vuC In Da Club 50 Cent
5KY7zgFeH2GWoL1zP9mME6 Get Low - Street Lil Jon & The East Side Boyz
7uKcScNXuO3MWw6LowBjW1 One, Two Step Ciara
70cTMpcgWMcR18t9MRJFjB I Gotta Feeling The Black Eyed Peas
2CEgGE6aESpnmtfiZwYlbV Dynamite Taio Cruz
4E5P1XyAFtrjpiIxkydly4 Replay Iyaz
0WqIKmW4BTrj3eJFmnCKMv Crazy In Love Beyoncé
2IpGdrWvIZipmaxo1YRxw5 Bottoms Up - feat. Nicki Minaj Trey Songz
0CAfXk7DXMnon4gLudAp7J Low (feat T-Pain) - Feat T-Pain Album Version Flo Rida
3VzJE6yGuj8fDExUh6TLnc Candy Shop 50 Cent
0k6HUzaRHpQ3eEWr1C7Esh Ms. New Booty - feat. Ying Yang Twins and Mr. ColliPark Bubba Sparxxx
557un1HgwYMuqfWGSTmnxw Single Ladies (Put a Ring on It) Beyoncé
5Q0Nhxo0l2bP3pNjpGJwV1 Party In The U.S.A. Miley Cyrus
1ARJhjuI6TNYZCxYygFQ4F Right Now (Na Na Na) Akon
5i66xrvSh1MjjyDd6zcwgj Umbrella Rihanna
1dzQoRqT5ucxXVaAhTcT0J Just Dance Lady Gaga
6RcQOut9fWL6FSqeIr5M1r

In [151]:
def extract_track_uris(playlist):
    return [track['track_uri'] for track in playlist['tracks']]

playlist = jsondata['playlists'][0]
track_uris = extract_track_uris(playlist)
track_uris

['spotify:track:0UaMYEvWZi0ZqiDOoHU3YI',
 'spotify:track:6I9VzXrHxO9rA9A5euc8Ak',
 'spotify:track:0WqIKmW4BTrj3eJFmnCKMv',
 'spotify:track:1AWQoqb9bSvzTjaLralEkT',
 'spotify:track:1lzr43nnXAijIGYnCT8M8H',
 'spotify:track:0XUfyU2QviPAs6bxSpXYG4',
 'spotify:track:68vgtRHr7iZHpzGpon6Jlo',
 'spotify:track:3BxWKCI06eQ5Od8TY2JBeA',
 'spotify:track:7H6ev70Weq6DdpZyyTmUXk',
 'spotify:track:2PpruBYCo4H7WOBJ7Q2EwM',
 'spotify:track:2gam98EZKrF9XuOkU13ApN',
 'spotify:track:4Y45aqo9QMa57rDsAJv40A',
 'spotify:track:1HwpWwa6bnqqRhK8agG4RS',
 'spotify:track:20ORwCJusz4KS2PbTPVNKo',
 'spotify:track:7k6IzwMGpxnRghE7YosnXT',
 'spotify:track:1Bv0Yl01xBDZD4OQP93fyl',
 'spotify:track:4omisSlTk6Dsq2iQD7MA07',
 'spotify:track:7xYnUQigPoIDAMPVK79NEq',
 'spotify:track:6d8A5sAx9TfdeseDvfWNHd',
 'spotify:track:4pmc2AxSEq6g7hPVlJCPyP',
 'spotify:track:215JYyyUnrJ98NK3KEwu6d',
 'spotify:track:0uqPG793dkDDN7sCUJJIVC',
 'spotify:track:19Js5ypV6JKn4DMExHQbGc',
 'spotify:track:1JURww012QnWAw0zZXi6Aa',
 'spotify:track:

In [161]:
def split_seed_ground_truth(track_uris, seed_size=1):
    seed_tracks = track_uris[:seed_size]        # first N tracks
    ground_truth = track_uris[seed_size:]       # remaining tracks
    return seed_tracks, ground_truth

seed_tracks, ground_truth = split_seed_ground_truth(track_uris)    


def format_track_uri(track_uri):
    return track_uri.split(':')[-1]

seed_tracks = [format_track_uri(t) for t in seed_tracks]
ground_truth = [format_track_uri(t) for t in ground_truth]



In [162]:
import numpy as np
from collections import defaultdict

def recommend_tracks(seed_tracks, X, knn_model, track_to_idx, idx_to_track, track_to_details, top_k=10):
    """
    Recommend tracks given seed track(s) using item-based KNN.

    Args:
        seed_tracks: list of track URIs (seed tracks from a playlist)
        X: sparse playlist x track matrix
        knn_model: trained sklearn NearestNeighbors on X.T (item-based)
        track_to_idx: dict mapping track_uri -> track_idx
        idx_to_track: dict mapping track_idx -> track_uri
        track_to_details: dict mapping track_uri -> (track_name, track_artist)
        top_k: number of tracks to recommend

    Returns:
        recommended_tracks: list of dicts with keys: 'track_uri', 'track_name', 'track_artist'
    """
    # Collect neighbors for all seed tracks
    neighbor_scores = defaultdict(float) 

    for track_uri in seed_tracks:
        if track_uri not in track_to_idx:
            continue  # skip unknown tracks

        track_idx = track_to_idx[track_uri]

        distances, neighbors = knn_model.kneighbors(X.T[track_idx], n_neighbors=top_k + len(seed_tracks))

        neighbors = neighbors[0]
        distances = distances[0]

        # Convert distance to similarity (cosine: similarity = 1 - distance)
        for neighbor_idx, distance in zip(neighbors, distances):
            if idx_to_track[neighbor_idx] in seed_tracks:
                continue  # skip seed tracks
            similarity = 1 - distance
            neighbor_scores[neighbor_idx] += similarity

    # Rank neighbors by aggregated similarity
    ranked_neighbors = sorted(neighbor_scores.items(), key=lambda x: x[1], reverse=True)

    # Take top_k
    top_neighbors = ranked_neighbors[:top_k]

    # Decode track info
    recommended_tracks = []
    for idx, score in top_neighbors:
        track_uri = idx_to_track[idx]
        track_name, track_artist = track_to_details.get(track_uri, ("Unknown", "Unknown"))
        recommended_tracks.append({
            "track_uri": track_uri,
            "track_name": track_name,
            "track_artist": track_artist,
            "score": score
        })

    return recommended_tracks

In [ ]:
recommended = recommend_tracks(seed_tracks, X, knn_model, track_to_id, id_to_track, track_to_details, top_k=10)
recommended_uris = [r['track_uri'] for r in recommended]



In [165]:
recommended

[{'track_uri': '4OjuzM3rTWp3q60mEldNSF',
  'track_name': 'Hit the Quan - Original Version',
  'track_artist': 'iLoveMemphis',
  'score': np.float64(0.6123724356957946)},
 {'track_uri': '4wH4dJgrsxONID6KS2tDQM',
  'track_name': 'Maneater',
  'track_artist': 'Nelly Furtado',
  'score': np.float64(0.5773502691896258)},
 {'track_uri': '0z9hbAY5d0RhmvZr3DWMPK',
  'track_name': 'Do It Like Me',
  'track_artist': 'DLOW',
  'score': np.float64(0.4714045207910318)},
 {'track_uri': '6m59VvDUi0UQsB2eZ9wVbH',
  'track_name': 'Poison',
  'track_artist': 'Bell Biv DeVoe',
  'score': np.float64(0.4714045207910318)},
 {'track_uri': '1b7vg5T9YKR3NNqXfBYRF7',
  'track_name': 'Check Yes Juliet',
  'track_artist': 'We The Kings',
  'score': np.float64(0.4714045207910318)},
 {'track_uri': '63IIUzd6eJBfCGIcF8MFnJ',
  'track_name': 'Way Away',
  'track_artist': 'Yellowcard',
  'score': np.float64(0.4714045207910318)},
 {'track_uri': '7xYnUQigPoIDAMPVK79NEq',
  'track_name': 'Run It!',
  'track_artist': 'Chri

### Model Evaluation

In [169]:
import numpy as np

def precision_at_k(recommended, ground_truth, k=10):
    top_k = recommended[:k]
    hits = sum([1 for t in top_k if t in ground_truth])
    return hits / k

def recall_at_k(recommended, ground_truth, k=10):
    top_k = recommended[:k]
    hits = sum([1 for t in top_k if t in ground_truth])
    return hits / len(ground_truth) if len(ground_truth) > 0 else 0.0

def ndcg_at_k(recommended, ground_truth, k=10):
    dcg = 0.0
    for i, t in enumerate(recommended[:k]):
        if t in ground_truth:
            dcg += 1 / np.log2(i + 2)
    ideal_hits = min(len(ground_truth), k)
    idcg = sum([1 / np.log2(i + 2) for i in range(ideal_hits)])
    return dcg / idcg if idcg > 0 else 0.0

def hit_rate_at_k(recommended, ground_truth, k=10):
    top_k = recommended[:k]
    return 1.0 if any(t in ground_truth for t in top_k) else 0.0

def mrr_at_k(recommended, ground_truth, k=10):
    top_k = recommended[:k]
    for i, t in enumerate(top_k):
        if t in ground_truth:
            return 1.0 / (i + 1)
    return 0.0


In [171]:
precision_list = []
recall_list = []
ndcg_list = []
hit_rate_list = []
mrr_list = []

unique_recommended_tracks = set()

for playlist in jsondata['playlists']:
    track_uris = [t['track_uri'].split(':')[-1] for t in playlist['tracks']]
    if len(track_uris) < 2:
        continue

    seed_tracks = track_uris[:-1]
    ground_truth = track_uris[-1:]

    seed_tracks = [t for t in seed_tracks if t in track_to_id]
    ground_truth = [t for t in ground_truth if t in track_to_id]
    if not seed_tracks or not ground_truth:
        continue

    recommended = recommend_tracks(seed_tracks, X, knn_model, track_to_id, id_to_track, track_to_details, top_k=10)
    recommended_uris = [r['track_uri'] for r in recommended]

    unique_recommended_tracks.update(recommended_uris)

    precision_list.append(precision_at_k(recommended_uris, ground_truth, k=10))
    recall_list.append(recall_at_k(recommended_uris, ground_truth, k=10))
    ndcg_list.append(ndcg_at_k(recommended_uris, ground_truth, k=10))
    hit_rate_list.append(hit_rate_at_k(recommended_uris, ground_truth, k=10))
    mrr_list.append(mrr_at_k(recommended_uris, ground_truth, k=10))

# Coverage: fraction of unique tracks recommended out of total tracks
coverage = len(unique_recommended_tracks) / len(track_to_id)

print("Average Precision@10:", np.mean(precision_list))
print("Average Recall@10:", np.mean(recall_list))
print("Average NDCG@10:", np.mean(ndcg_list))
print("Average Hit Rate@10:", np.mean(hit_rate_list))
print("Average MRR@10:", np.mean(mrr_list))
print("Coverage:", coverage)


Average Precision@10: 0.08610000000000001
Average Recall@10: 0.861
Average NDCG@10: 0.8358226638643782
Average Hit Rate@10: 0.861
Average MRR@10: 0.8280186507936508
Coverage: 0.1909241355282641
